In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/544 Project/"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import random
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import torch
from huggingface_hub import login
from torchinfo import summary
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
from peft import PeftModel

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# ADD Hugging Face API KEY instead of HF_TOKEN
login(token="HF_TOKEN")

In [ ]:
# ADD "base" or "lora" instead of "MODEL_SOURCE"
MODEL_SOURCE = "MODEL_SOURCE"
# ADD "Qwen/Qwen2.5-3B-Instruct" or "google/gemma-3-4b-it" instead of "MODEL_ID"
BASE_MODEL_ID = "MODEL_ID"

In [ ]:
# ADD "train/gemma/model_files/model" or "train/qwen/model_files/model" instead of "LORA_MODEL_PATH"
LORA_MODEL_PATH = "./PATH/TO/LORA/MODEL"
# ADD "train/gemma/model_files/tokenizer" or "train/qwen/model_files/tokenizer" instead of "LORA_TOKENIZER_PATH"
LORA_TOKENIZER_PATH = "./PATH/TO/LORA/TOKENIZER"

In [ ]:
DATA_PATHS = {
    "qa": "./data/test/halueval_qa_data.xlsx",
    "dialogue": "./data/test/halueval_dialogue_data.xlsx",
    "summarization": "./data/test/halueval_summarization_data.xlsx",
}

In [ ]:
# ADD "qa" or "dialogue" or "summarization" instead of "TASK"
TASK = "TASK"

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [ ]:
OUTPUT_PATH = f"./test_results/{MODEL_SOURCE}_{BASE_MODEL_ID}_{TASK}_evaluation_results.json"

In [ ]:
MAX_SAMPLES = None
MAX_NEW_TOKENS = 8
BATCH_SIZE = 32 

In [ ]:
def load_instruction(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [ ]:
INSTRUCTION_PATHS = {
    "qa": "./data/test/instruction_files/qa_evaluation_instruction.txt",
    "dialogue": "./data/test/instruction_files/dialogue_evaluation_instruction.txt",
    "summarization": "./data/test/instruction_files/summarization_evaluation_instruction.txt",
}

In [ ]:
INSTRUCTIONS = {}
for task, path in INSTRUCTION_PATHS.items():
    INSTRUCTIONS[task] = load_instruction(path)
    print(f"[{task}] Loaded from {path}  ({len(INSTRUCTIONS[task])} chars)")

In [ ]:
def build_chat_messages(task: str, row: dict, instruction: str) -> list:
    if task == "qa":
        user_content = (
            instruction
            + "\n\n#Question#: " + row["question"]
            + "\n#Answer#: " + row["answer"]
            + "\n#Your Judgement#:"
        )
    elif task == "dialogue":
        user_content = (
            instruction
            + "\n\n#Dialogue History#: " + row["dialogue_history"]
            + "\n#Response#: " + row["response"]
            + "\n#Your Judgement#:"
        )
    elif task == "summarization":
        user_content = (
            instruction
            + "\n\n#Document#: " + row["document"]
            + "\n#Summary#: " + row["summary"]
            + "\n#Your Judgement#:"
        )
    else:
        raise ValueError(f"Unknown task: {task}")

    return [{"role": "user", "content": user_content}]

In [ ]:
USE_4BIT = False
bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

In [ ]:
if MODEL_SOURCE == "lora":
    tokenizer = AutoTokenizer.from_pretrained(LORA_TOKENIZER_PATH, trust_remote_code=True)
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [ ]:
print(f"Loading base model: {BASE_MODEL_ID} ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config if USE_4BIT else None,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

In [ ]:
if MODEL_SOURCE == "lora":
    print(f"Loading LoRA adapter from: {LORA_MODEL_PATH} ...")
    model = PeftModel.from_pretrained(base_model, LORA_MODEL_PATH)
    model = model.merge_and_unload()
    print("LoRA weights merged.")
else:
    model = base_model

In [ ]:
model.eval()
print("Model ready.", f" Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def get_judgement(messages: list) -> str:
    has_chat_template = (
        hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    )

    if has_chat_template:
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        parts = []
        for m in messages:
            if m["role"] == "system":
                parts.append(m["content"])
            elif m["role"] == "user":
                parts.append(m["content"])
        input_text = "\n\n".join(parts)

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=2048).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response

In [ ]:
def parse_judgement(raw: str) -> str:
    cleaned = raw.replace(".", "").strip()
    has_yes = "Yes" in cleaned
    has_no  = "No"  in cleaned

    if (has_yes and has_no) or (not has_yes and not has_no):
        return "failed!"
    elif has_yes:
        return "Yes"
    else:
        return "No"

In [ ]:
data_path = DATA_PATHS[TASK]
df = pd.read_excel(data_path)
print(f"Loaded {len(df)} rows from {data_path}")
print(df.columns.tolist())
df.head(2)

In [ ]:
if MAX_SAMPLES is not None and MAX_SAMPLES < len(df):
    df = df.sample(MAX_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Subsampled to {len(df)} rows.")

In [ ]:
instruction = INSTRUCTIONS[TASK]
rng = random.Random(RANDOM_SEED)
eval_rows = []

In [ ]:
for _, row in df.iterrows():
    use_hallucinated = rng.random() > 0.5

    if TASK == "qa":
        base = {
            "knowledge": row["knowledge"],
            "question":  row["question"],
        }
        if use_hallucinated:
            base["answer"] = row["hallucinated_answer"]
            base["ground_truth"] = "Yes"
        else:
            base["answer"] = row["right_answer"]
            base["ground_truth"] = "No"

    elif TASK == "dialogue":
        base = {
            "knowledge": row["knowledge"],
            "dialogue_history": row["dialogue_history"],
        }
        if use_hallucinated:
            base["response"] = row["hallucinated_response"]
            base["ground_truth"] = "Yes"
        else:
            base["response"] = row["right_response"]
            base["ground_truth"] = "No"

    elif TASK == "summarization":
        base = {"document": row["document"]}
        if use_hallucinated:
            base["summary"] = row["hallucinated_summary"]
            base["ground_truth"] = "Yes"
        else:
            base["summary"] = row["right_summary"]
            base["ground_truth"] = "No"

    eval_rows.append(base)

In [ ]:
yes_count = sum(1 for r in eval_rows if r["ground_truth"] == "Yes")
print(f"Evaluation set: {len(eval_rows)} samples | hallucinated={yes_count} correct={len(eval_rows)-yes_count}")

In [ ]:
output_file = Path(OUTPUT_PATH)
output_file.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
correct = 0
incorrect = 0
failed = 0
results = []

In [ ]:
with output_file.open("w", encoding="utf-8") as fout:
    for i, row in enumerate(tqdm(eval_rows, desc=f"Evaluating [{TASK}]")):
        messages = build_chat_messages(TASK, row, instruction)

        try:
            raw = get_judgement(messages)
        except Exception as e:
            print(f"\nSample {i} ERROR: {e}")
            judgement = "failed!"
            raw = ""
        else:
            judgement = parse_judgement(raw)

        ground_truth = row["ground_truth"]
        gen = {**row, "raw_output": raw, "judgement": judgement}

        if judgement == "failed!":
            failed    += 1
            incorrect += 1
        elif judgement == ground_truth:
            correct += 1
        else:
            incorrect += 1

        results.append(gen)
        fout.write(json.dumps(gen, ensure_ascii=False) + "\n")

print(f"\nDone. {correct} correct | {incorrect} incorrect (incl. {failed} failed) | Total {len(eval_rows)}")

In [ ]:
valid = [r for r in results if r["judgement"] != "failed!"]
y_true = [r["ground_truth"] for r in valid]
y_pred = [r["judgement"]    for r in valid]
total   = len(results)
n_valid = len(valid)
n_fail  = total - n_valid

In [ ]:
acc_all   = correct / total if total > 0 else 0.0
acc_valid = accuracy_score(y_true, y_pred) if n_valid > 0 else 0.0

In [ ]:
pos_label = "Yes"
prec = precision_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
rec = recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
f1 = f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0)

In [ ]:
print(f"Task : {TASK}")
print(f"Model : {BASE_MODEL_ID}")
if MODEL_SOURCE == 'lora':
    print(f"LoRA adapter: {LORA_MODEL_PATH}")
print(f"Total samples     : {total}")
print(f"Valid (non-failed): {n_valid}")
print(f"Failed : {n_fail}")
print(f"Accuracy (all): {acc_all:.4f}")
print(f"Accuracy (valid): {acc_valid:.4f}")
print(f"Precision (Yes): {prec:.4f}")
print(f"Recall (Yes): {rec:.4f}")
print(f"F1 (Yes): {f1:.4f}")

In [ ]:
if n_valid > 0:
    print("\nClassification Report (valid samples):")
    print(classification_report(y_true, y_pred, digits=4))
    print("Confusion Matrix (rows=true, cols=pred):")
    labels = ["Yes", "No"]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=[f"True_{l}" for l in labels], columns=[f"Pred_{l}" for l in labels])
    print(cm_df)

In [ ]:
summary = {
    "task": TASK,
    "model": BASE_MODEL_ID,
    "model_source": MODEL_SOURCE,
    # "lora_path": LORA_MODEL_PATH if MODEL_SOURCE == "lora" else "",
    "total": total,
    "valid": n_valid,
    "failed": n_fail,
    "correct": correct,
    "incorrect": incorrect,
    "accuracy_all": round(acc_all, 4),
    "accuracy_valid": round(acc_valid, 4),
    "precision_yes": round(prec, 4),
    "recall_yes": round(rec, 4),
    "f1_yes": round(f1, 4),
}

In [ ]:
summary_path = "./test_results/results_benchmark_lora_selfconsistency.xlsx"
summary_df = pd.DataFrame([summary])

In [ ]:
if Path(summary_path).exists():
    existing = pd.read_csv(summary_path)
    summary_df = pd.concat([existing, summary_df], ignore_index=True)

In [ ]:
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved/appended to {summary_path}")
summary_df